In [0]:
%run "../00_Setup_Config/project_config"

In [0]:
# Databricks notebook source
import dlt
from pyspark.sql.functions import col, sum, count, avg, count_distinct, round, current_timestamp, lit, broadcast

spark.conf.set("spark.sql.shuffle.partitions", "400")

# --- DOMAIN 5: AUDIT LOGGING HELPER ---
def add_audit_metadata(df):
    """Adds traceability columns to meet Domain 5 governance requirements."""
    return df.withColumn("ingestion_timestamp", current_timestamp()) \
             .withColumn("processed_by_user", lit("dlt_pipeline_service"))

# --- CONSOLIDATED GOLD TABLE ---
@dlt.table(
    name="gold_layer.product_performance",
    comment="Gold Layer KPI: Joined from Silver Events and Gold SCD2 Dimensions",
    table_properties={
        "quality": "gold",
        "delta.autoOptimize.optimizeWrite": "true", 
        "pipelines.autoOptimize.zOrderCols": "brand"
    }
)
def calculate_product_performance():
    # 1. READ FROM SILVER LAYER
    # Using the full path to strictly follow your structure
    fact_sales = dlt.read("gold_layer.fact_sales")
    
    # 2. READ FROM GOLD LAYER (SCD2 table)
    dim_products = dlt.read("gold_layer.dim_products_scd2")

    joined_df = fact_sales.join(broadcast(dim_products), on="product_id", how="inner")
    
    # 3. JOIN & AGGREGATE
    # Explicitly reference dataframes to resolve the 'price' ambiguity
    kpi_df = (
        joined_df.groupBy(dim_products["brand"])
        # fact_sales.join(dim_products, on="product_id", how="inner")
        # .groupBy(dim_products["brand"])
        .agg(
            round(sum(fact_sales["price"]), 2).alias("actual_revenue"),
            round(avg(dim_products["price"]), 2).alias("historical_catalog_price"), # Comma was likely missing here
            count(fact_sales["product_id"]).alias("total_units_sold"),
            count_distinct(fact_sales["user_session"]).alias("total_sessions")
        )
    )
    
    # 4. RETURN WITH AUDIT METADATA
    return add_audit_metadata(kpi_df)

In [0]:
# Table 2: Customer Journey Metrics
@dlt.table(
    name="gold_layer.customer_metrics",
    comment="Gold Layer: Tracks user behavior and session-level conversion behavior",
    table_properties={"quality": "gold"}
)
def customer_journey_metrics():
    # Reads from Silver to include all interactions (views, carts, purchases)
    return (
        dlt.read("ecommerce_analytics_dev.silver_layer.events_cleaned")
        .groupBy("user_id")
        .agg(
            count_distinct("user_session").alias("total_sessions"),
            count(col("event_type")).alias("total_interactions"),
            # Rounding for clean dashboard presentation
            round(avg(col("price")), 2).alias("avg_interaction_value")
        )
    )